# Standardized Robustness Pipeline: Poisoning + Evasion (Label Flip, HSJ, Boundary, ZOO)

This notebook standardizes experiments across your 5 models:

- **Models** (your existing runners):
  - `run_logreg`, `run_neuralnet`, `run_randomforest`, `run_svm`, `run_xgboost`
- **Poisoning**:
  - Label Flip (training-time)
- **Evasion**:
  - HopSkipJump (HSJ), BoundaryAttack, ZooAttack (evaluation-time)

## Key design choices (matches your constraints)

- **One canonical train/test split** made once from the full dataset.
- **No detectors**.
- **Never train on test data** (clean or adversarial). Training uses *train split only*.
- **Poisoning** modifies **only the train split** labels.
- **Adversarial training (Option B)** is implemented by:
  1. training the model on the current train dataset (clean or augmented)
  2. generating adversarial examples **from train only** using ART against the current model
  3. appending those adversarial rows (with correct labels) to the train dataset
  4. re-training and re-evaluating
- **Adversarial examples are regenerated each round.**

## Practical runtime notes

Decision-based attacks (HSJ/Boundary) and ZOO can be slow.
This notebook attacks a configurable **subset** for evaluation and for adversarial training augmentation.
You can scale up, but “full dataset HSJ/ZOO” may take a long time.

---


In [28]:
# ===== Imports =====
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Your model runners (expected to exist in your project)
# NOTE: If running this notebook outside your project root, set PYTHONPATH accordingly.
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost

# ART: import SklearnClassifier in a way that avoids importing KerasClassifier
# (some ART installs break if keras-related utils are missing).
try:
    from art.estimators.classification.scikitlearn import SklearnClassifier
except Exception:
    from art.estimators.classification import SklearnClassifier

from xgboost import XGBClassifier

# ART XGBoost wrapper (path varies by ART version)
try:
    from art.estimators.classification.xgboost import XGBoostClassifier
except Exception:
    from art.estimators.classification import XGBoostClassifier



from art.attacks.evasion import HopSkipJump, BoundaryAttack, ZooAttack

RNG = np.random.default_rng(42)



In [29]:
# ===== Configuration =====
LABEL_COL = "anomaly"

# Columns often present in your project; will be dropped from features if they exist
DROP_COLS = {"anomaly", "timestamp", "channel", "label", "segment", "train"}

# Train/test split (canonical, used for ALL models & attacks)
TEST_SIZE = 0.2
SPLIT_RANDOM_STATE = 42

# Attack budget knobs (keep realistic)
EVAL_ATTACK_SAMPLES = 10      # number of test points to attack per model/attack
TRAIN_ADV_SAMPLES = 10        # number of train points to adversarially augment per round
ROUNDS = 1                     # adversarial training rounds (Option B)

# Label flip poisoning knobs
LABEL_FLIP_RATES = [0.05, 0.10, 0.20]

# Attack configs (tune for your compute)
HSJ_KWARGS = dict(max_iter=20, max_eval=5000, init_eval=50, init_size=10, targeted=False, norm=2)
HSJ_TRAIN_KWARGS = dict(max_iter=10, max_eval=2000, init_eval=25, init_size=10, targeted=False, norm=2)

BOUNDARY_KWARGS = dict(targeted=False, max_iter=2000, init_size=10)  # Boundary can be slow
ZOO_KWARGS = dict(max_iter=10, binary_search_steps=1, nb_parallel=1, batch_size=1)  # keep light


import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names*")



In [30]:
# ===== Load dataset and create canonical split =====

df = pd.read_csv("CSVs\\newDataset.csv")

# Build feature columns (keep only non-label columns; drop known metadata columns if present)
feature_cols = [c for c in df.columns if c not in DROP_COLS and c != LABEL_COL]

# Handle missing values
from sklearn.impute import SimpleImputer

# Force numeric (common cause: strings -> NaN)
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols]).astype(np.float32)

# Kill any leftover Inf just in case
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

y_all = df[LABEL_COL].to_numpy(dtype=int)
_, y_all = np.unique(y_all, return_inverse=True)

print("X_all nonfinite:", np.sum(~np.isfinite(X_all)))

# Keep only numeric features for attacks; if you have categorical columns, encode them before this notebook.
X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df[LABEL_COL].to_numpy(dtype=int)

# Ensure labels are 0..K-1
_, y_all = np.unique(y_all, return_inverse=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE, stratify=y_all
)

print("Train:", X_train.shape, y_train.shape, "classes:", len(np.unique(y_train)))
print("Test :", X_test.shape, y_test.shape)


X_all nonfinite: 0
Train: (1698, 20) (1698,) classes: 2
Test : (425, 20) (425,)


In [31]:
# ===== Helpers: build train-only CSVs for your existing model runners =====
# Your run_best_model functions read from CSV and do their own internal split.
# To enforce "never train on test", we give them TRAIN-ONLY CSVs.
# We then evaluate the returned trained pipeline on our held-out X_test/y_test.

WORK_DIR = "StandardizedRuns"
os.makedirs(WORK_DIR, exist_ok=True)

def make_train_df_from_arrays(X_tr: np.ndarray, y_tr: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(X_tr, columns=feature_cols)
    out[LABEL_COL] = y_tr
    return out

def save_train_csv(df_train: pd.DataFrame, name: str) -> str:
    path = os.path.join(WORK_DIR, name)
    df_train.to_csv(path, index=False)
    return path

def fit_model_with_runner(model_name: str, runner_fn, train_csv_path: str):
    # Use a more reasonable internal split than some module defaults.
    # Many of your modules default to test_size=0.80 (very large). Override to 0.2.
    pipe, _internal_X_test, _internal_y_test = runner_fn(
        path=train_csv_path,
        test_size=0.2,
        random_state=SPLIT_RANDOM_STATE
    )
    return pipe

def eval_clean(pipe, X: np.ndarray, y: np.ndarray) -> float:
    X_df = pd.DataFrame(X, columns=feature_cols)  # preserve feature names
    y_pred = pipe.predict(X_df)
    return float(accuracy_score(y, y_pred))



In [32]:
# ===== ART helpers =====
from sklearn.base import BaseEstimator, ClassifierMixin

class SafePredictWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, base_model, clip_values=None):
        self.base_model = base_model
        self.clip_values = clip_values

    def _clean(self, X):
        X = np.asarray(X, dtype=np.float32)
        # replace NaN/Inf
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        # optional clamp
        if self.clip_values is not None:
            lo, hi = self.clip_values
            if np.isfinite(lo) and np.isfinite(hi):
                X = np.clip(X, lo, hi)
        return X

    def predict(self, X):
        return self.base_model.predict(self._clean(X))

    def predict_proba(self, X):
        return self.base_model.predict_proba(self._clean(X))

    def __getattr__(self, name):
        # delegate anything else (e.g., classes_) to the underlying model
        return getattr(self.base_model, name)



def wrap_art(model, X_ref: np.ndarray, y_ref: np.ndarray):
    # NaN-safe bounds
    clip_values = (float(np.nanmin(X_ref)), float(np.nanmax(X_ref)))

    nb_classes = int(len(np.unique(y_ref)))
    input_shape = (X_ref.shape[1],)

    if isinstance(model, XGBClassifier):
        est = XGBoostClassifier(
            model=model,
            clip_values=clip_values,
            nb_classes=nb_classes,
        )
        if getattr(est, "input_shape", None) is None:
            est.input_shape = input_shape
        return est

    # For sklearn models (MLP, etc.)
    safe_model = model
    # Only wrap MLPClassifier, since it's the one hard-failing
    if model.__class__.__name__ == "MLPClassifier":
        safe_model = SafePredictWrapper(model, clip_values=clip_values)

    return SklearnClassifier(model=safe_model, clip_values=clip_values)


def unwrap_estimator(est):
    """Try to peel common wrappers (ART, sklearn Pipeline/meta-estimators) to find the core estimator."""
    seen = set()
    while est is not None and id(est) not in seen:
        seen.add(id(est))

        # ART sometimes stores the sklearn model here
        if hasattr(est, "_model"):
            est = getattr(est, "_model")
            continue

        # Your own wrapper pattern
        if hasattr(est, "base_model"):
            est = getattr(est, "base_model")
            continue

        # sklearn Pipeline
        if hasattr(est, "named_steps") and len(est.named_steps) > 0:
            est = list(est.named_steps.values())[-1]
            continue
        if hasattr(est, "steps") and len(est.steps) > 0:
            est = est.steps[-1][1]
            continue

        # common meta-estimators
        if hasattr(est, "estimator"):
            est = getattr(est, "estimator")
            continue
        if hasattr(est, "base_estimator"):
            est = getattr(est, "base_estimator")
            continue

        break
    return est

def is_mlp(est):
    core = unwrap_estimator(est)
    return core is not None and core.__class__.__name__ == "MLPClassifier"



def predict_labels_art(art_clf: SklearnClassifier, X: np.ndarray) -> np.ndarray:
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

def sample_subset(X: np.ndarray, y: np.ndarray, n: int, rng=RNG):
    if n >= len(X):
        return X, y
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

def attack_success_rate(y_true: np.ndarray, y_pred_clean: np.ndarray, y_pred_adv: np.ndarray) -> float:
    mask = (y_pred_clean == y_true)
    if mask.sum() == 0:
        return float('nan')
    return float((y_pred_adv[mask] != y_true[mask]).mean())

def eval_under_attack(attack_name: str, art_clf: SklearnClassifier, X_eval: np.ndarray, y_eval: np.ndarray):
    # Generate adversarial examples for evaluation subset and compute metrics.
    if attack_name == "HSJ":
        atk = HopSkipJump(classifier=art_clf, **HSJ_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)
    elif attack_name == "Boundary":
    # Robust skip for MLP (NN)
        est = getattr(art_clf, "model", None)
        if est is None:
            est = getattr(art_clf, "_model", None)
    
        if is_mlp(est):
            return np.nan, np.nan, np.nan, np.nan
    
        atk = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)


    elif attack_name == "ZOO":
        atk = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
        # ZOO often expects one-hot labels in some setups, but can work with integers depending on estimator.
        # We'll try integer labels first; if it errors, we fall back to simple one-hot.
        try:
            X_adv = atk.generate(x=X_eval, y=y_eval)
        except Exception:
            k = int(len(np.unique(y_train)))
            y_oh = np.zeros((len(y_eval), k), dtype=np.float32)
            y_oh[np.arange(len(y_eval)), y_eval.astype(int)] = 1.0
            X_adv = atk.generate(x=X_eval, y=y_oh)
    else:
        raise ValueError(f"Unknown attack: {attack_name}")

    X_adv = np.asarray(X_adv, dtype=np.float32)

    y_pred_clean = predict_labels_art(art_clf, X_eval)
    y_pred_adv = predict_labels_art(art_clf, X_adv)

    clean_acc = float(accuracy_score(y_eval, y_pred_clean))
    adv_acc = float(accuracy_score(y_eval, y_pred_adv))
    drop = clean_acc - adv_acc
    asr = attack_success_rate(y_eval, y_pred_clean, y_pred_adv)
    return clean_acc, adv_acc, drop, asr


In [33]:
# ===== Poisoning: Label Flip (train-only) =====

def label_flip(y: np.ndarray, flip_rate: float, rng=RNG) -> tuple[np.ndarray, dict]:
    y = np.asarray(y, dtype=int).copy()
    n = len(y)
    k = int(len(np.unique(y_train)))
    m = int(round(flip_rate * n))
    idx = rng.choice(n, size=m, replace=False)

    # Flip strategy for binary and multi-class
    if k == 2:
        y[idx] = 1 - y[idx]
    else:
        # For each selected index, pick a different class uniformly
        for i in idx:
            choices = [c for c in range(k) if c != y[i]]
            y[i] = rng.choice(choices)

    meta = {"flip_rate": float(flip_rate), "num_flipped": int(m), "indices": idx.tolist()}
    return y, meta


In [34]:
# ===== Standardized experiment runners =====

MODEL_RUNNERS = {
    "LogReg": run_logreg,
    "NeuralNet": run_neuralnet,
    "RandomForest": run_randomforest,
    "SVM": run_svm,
    # "XGBoost": run_xgboost,
}

EVASION_ATTACKS = ["HSJ", "Boundary", "ZOO"]

def run_baseline_and_evasion(model_name: str, runner_fn):
    rows = []

    # Train on clean TRAIN ONLY
    train_df = make_train_df_from_arrays(X_train, y_train)
    clean_train_path = save_train_csv(train_df, f"{model_name}_train_clean.csv")
    pipe = fit_model_with_runner(model_name, runner_fn, clean_train_path)

    # Clean eval (full held-out test)
    clean_test_acc = eval_clean(pipe, X_test, y_test)

    # Evasion eval on subset
    art_clf = wrap_art(pipe, X_train, y_train)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr = eval_under_attack(atk, art_clf, X_eval, y_eval)
        rows.append({
            "model": model_name,
            "phase": "baseline",
            "attack_type": "evasion",
            "attack": atk,
            "round": 0,
            "clean_test_acc": clean_test_acc,
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": 0.0,
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
        })

    return pipe, pd.DataFrame(rows)

def run_label_flip_poisoning(model_name: str, runner_fn, flip_rate: float):
    rows = []

    y_poison, meta = label_flip(y_train, flip_rate)
    train_df_poison = make_train_df_from_arrays(X_train, y_poison)
    poison_path = save_train_csv(train_df_poison, f"{model_name}_train_labelflip_{int(flip_rate*100)}.csv")

    pipe = fit_model_with_runner(model_name, runner_fn, poison_path)

    clean_test_acc = eval_clean(pipe, X_test, y_test)

    # Optional: evaluate evasion robustness after poisoning (often interesting)
    art_clf = wrap_art(pipe, X_train, y_train)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr = eval_under_attack(atk, art_clf, X_eval, y_eval)
        rows.append({
            "model": model_name,
            "phase": "poisoned",
            "attack_type": "poison+evasion",
            "attack": f"LabelFlip({flip_rate}) + {atk}",
            "round": 0,
            "clean_test_acc": clean_test_acc,
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": float(flip_rate),
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
            "poison_meta": json.dumps({k: v for k, v in meta.items() if k != "indices"}),
        })

    return pd.DataFrame(rows)

def run_adversarial_training(model_name: str, runner_fn, train_attack: str = "HSJ"):
    rows = []

    # Start with clean train
    X_tr = X_train.copy()
    y_tr = y_train.copy()
    augmented = 0

    for r in range(ROUNDS + 1):
        # Train current model on current train set (clean + accumulated adv)
        train_df = make_train_df_from_arrays(X_tr, y_tr)
        train_path = save_train_csv(train_df, f"{model_name}_train_advtrain_round{r}.csv")
        pipe = fit_model_with_runner(model_name, runner_fn, train_path)

        # Evaluate on full clean test
        clean_test_acc = eval_clean(pipe, X_test, y_test)

        # Evaluate evasion robustness on subset (HSJ/Boundary/ZOO)
        art_clf = wrap_art(pipe, X_train, y_train)

        X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

        for atk in EVASION_ATTACKS:
            cacc, aacc, drop, asr = eval_under_attack(atk, art_clf, X_eval, y_eval)
            rows.append({
                "model": model_name,
                "phase": "advtrain",
                "attack_type": "evasion",
                "attack": atk,
                "round": r,
                "clean_test_acc": clean_test_acc,
                "clean_acc_evalsubset": cacc,
                "adv_acc_evalsubset": aacc,
                "acc_drop_evalsubset": drop,
                "attack_success_rate": asr,
                "train_poison_rate": 0.0,
                "train_adv_augmented": augmented,
                "eval_attack_samples": len(X_eval),
            })

        if r == ROUNDS:
            break

        # Generate fresh adversarials from TRAIN subset only
        X_sub, y_sub = sample_subset(X_tr, y_tr, TRAIN_ADV_SAMPLES)

        if train_attack == "HSJ":
            atk_train = HopSkipJump(classifier=art_clf, **HSJ_TRAIN_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "Boundary":
            atk_train = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "ZOO":
            atk_train = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
            try:
                X_adv = atk_train.generate(x=X_sub, y=y_sub)
            except Exception:
                k = int(len(np.unique(y_train)))
                y_oh = np.zeros((len(y_sub), k), dtype=np.float32)
                y_oh[np.arange(len(y_sub)), y_sub.astype(int)] = 1.0
                X_adv = atk_train.generate(x=X_sub, y=y_oh)
        else:
            raise ValueError(train_attack)

        X_adv = np.asarray(X_adv, dtype=np.float32)

        # clamp to the same bounds you gave ART
        clip_min, clip_max = art_clf.clip_values
        X_adv = np.clip(X_adv, clip_min, clip_max)

        # last-resort cleanup (prevents sklearn from crashing)
        X_adv = np.nan_to_num(X_adv, nan=clip_min, posinf=clip_max, neginf=clip_min)

        # Append with correct labels
        X_tr = np.vstack([X_tr, X_adv]).astype(np.float32)
        y_tr = np.concatenate([y_tr, y_sub]).astype(int)
        augmented += len(X_adv)

        print(f"[{model_name}] adv-train round {r} -> {r+1}: +{len(X_adv)} using {train_attack}, train size={len(X_tr)}")

    return pd.DataFrame(rows)


In [35]:
# ===== XGB Helpers =====
from sklearn.pipeline import Pipeline

def unwrap_xgb(model):
    if isinstance(model, Pipeline):
        # last step model
        return model.steps[-1][1]
    return model

def wrap_art(model, X_ref: np.ndarray, y_ref: np.ndarray):
    clip_values = (float(np.min(X_ref)), float(np.max(X_ref)))
    nb_classes = int(len(np.unique(y_ref)))
    input_shape = (X_ref.shape[1],)

    if isinstance(model, XGBClassifier):
        return XGBoostClassifier(
            model=model,
            clip_values=clip_values,
            nb_classes=nb_classes,
            input_shape=input_shape,   # <-- critical for HSJ
        )

    return SklearnClassifier(model=model, clip_values=clip_values)



In [36]:
# ===== Run the full standardized suite =====

all_rows = []

for model_name, runner_fn in MODEL_RUNNERS.items():
    print("\n" + "="*80)
    print("MODEL:", model_name)
    print("="*80)

    # Baseline + evasion attacks
    _pipe, df_base = run_baseline_and_evasion(model_name, runner_fn)
    all_rows.append(df_base)

    # Poisoning: label flip rates
    for rate in LABEL_FLIP_RATES:
        df_poison = run_label_flip_poisoning(model_name, runner_fn, rate)
        all_rows.append(df_poison)

    # Option B: adversarial training (default HSJ) - comment out if too slow
    df_advtrain = run_adversarial_training(model_name, runner_fn, train_attack="HSJ")
    all_rows.append(df_advtrain)

results_df = pd.concat(all_rows, ignore_index=True)
results_df



MODEL: LogReg
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.982)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       271
           1       0.88      0.83      0.85        69

    accuracy                           0.94       340
   macro avg       0.92      0.90      0.91       340
weighted avg       0.94      0.94      0.94       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     263       8
True 1      12      57

AUC: 0.982

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 10/10 [00:00<00:00, 499.56it/s]


Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.836)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       261
           1       0.74      0.63      0.68        79

    accuracy                           0.86       340
   macro avg       0.81      0.78      0.80       340
weighted avg       0.86      0.86      0.86       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     243      18
True 1      29      50

AUC: 0.836

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 10/10 [00:00<00:00, 475.76it/s]


Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.789)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.86      0.91      0.88       253
           1       0.68      0.59      0.63        87

    accuracy                           0.82       340
   macro avg       0.77      0.75      0.76       340
weighted avg       0.82      0.82      0.82       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     229      24
True 1      36      51

AUC: 0.789

=== All results and summaries saved successfully ===


...


Saved last run results to Results/RandomForestResults\results_randomforest.csv
Saved ROC data to Results/RandomForestResults\roc_randomforest_clean.csv (AUC = 0.927)
Saved summary (AUC + Confusion Matrix) to Results/RandomForestResults\randomforest_summary.csv

=== Random Forest Test Set Report ===
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       260
           1       0.94      0.82      0.88        80

    accuracy                           0.95       340
   macro avg       0.95      0.90      0.92       340
weighted avg       0.95      0.95      0.95       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     256       4
True 1      14      66

AUC: 0.927

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 10/10 [00:00<00:00, 10.60it/s]



Saved last run results to Results/RandomForestResults\results_randomforest.csv
Saved ROC data to Results/RandomForestResults\roc_randomforest_clean.csv (AUC = 0.832)
Saved summary (AUC + Confusion Matrix) to Results/RandomForestResults\randomforest_summary.csv

=== Random Forest Test Set Report ===
              precision    recall  f1-score   support

           0       0.86      0.96      0.91       251
           1       0.84      0.57      0.68        89

    accuracy                           0.86       340
   macro avg       0.85      0.77      0.79       340
weighted avg       0.86      0.86      0.85       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     241      10
True 1      38      51

AUC: 0.832

=== All results and summaries saved successfully ===


Boundary attack:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\janst\AppData\Local\Programs\Python\Python310\lib\site-packages\art\attacks\evasion\boundary.py:351: RuntimeWarning: overflow encountered in multiply
  perturb *= delta * np.linalg.norm(original_sample - current_sample)
c:\Users\janst\AppData\Local\Programs\Python\Python310\lib\site-packages\art\attacks\evasion\boundary.py:360: RuntimeWarning: invalid value encountered in subtract
  perturb_flat -= np.dot(perturb_flat, direction_flat.T) * direction_flat
Boundary attack:   0%|          | 0/10 [00:50<?, ?it/s]


OverflowError: (34, 'Result too large')

In [ ]:
# ===== Save results and quick pivots =====

out_csv = os.path.join(WORK_DIR, "standardized_results.csv")
results_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# Quick pivot: baseline evasion drops
pivot = results_df[results_df["phase"].isin(["baseline", "advtrain"])].pivot_table(
    index=["model", "phase", "round"],
    columns=["attack"],
    values=["clean_test_acc", "adv_acc_evalsubset", "acc_drop_evalsubset"],
    aggfunc="mean"
)
pivot


Saved: StandardizedRuns\standardized_results.csv


acc_drop_evalsubset adv_acc_evalsubset  \
attack                                      HSJ                HSJ   
model        phase    round                                          
LogReg       advtrain 0                     0.9                0.1   
                      1                     0.6                0.3   
             baseline 0                     0.7                0.2   
NeuralNet    advtrain 0                     0.3                0.7   
                      1                     1.0                0.0   
             baseline 0                     0.4                0.5   
RandomForest advtrain 0                     0.3                0.7   
                      1                     0.1                0.8   
             baseline 0                     0.1                0.8   
SVM          advtrain 0                     0.5                0.5   
                      1                     0.9                0.0   
             baseline 0                     0.3                0.5   

                            clean_test_acc  
attack                                 HSJ  
model        phase    round                 
LogReg       advtrain 0           0.931765  
                      1           0.929412  
             baseline 0           0.931765  
NeuralNet    advtrain 0           0.957647  
                      1           0.948235  
             baseline 0           0.957647  
RandomForest advtrain 0           0.950588  
                      1           0.941176  
             baseline 0           0.950588  
SVM          advtrain 0           0.924706  
                      1           0.912941  
             baseline 0           0.924706